# Gemma clinical training: token and answer truncation audit

This audit measures the **actual training task** in `main.ipynb`: first 1,500 training rows, dataset system instruction folded into the user turn, then the original assistant answer. It does not add the baseline's JSON schema prompt, load model weights, train, or call a judge. A CPU runtime is enough.

Run top to bottom in VS Code/Colab. Enter a Hugging Face read token for the account that accepted the Gemma license when prompted. Saved baseline artifacts pin model/dataset revisions when available; otherwise revisions are resolved and recorded. Outputs go to `training_token_audit/`. Download that folder after running.

The truncation simulation assumes one independently tokenized text per row, **no packing**, and the tokenizer's configured truncation side. It reproduces `tokenizer(text)` on the training notebook's formatted strings, including default special-token insertion. Installed TRL versions may preprocess differently: this is an explicit tokenization audit, not a claim about unseen trainer labels. The final optional check can inspect a real trainer before training.


In [3]:
%pip install -q "transformers==4.57.6" "datasets>=3,<5" huggingface_hub pandas


In [4]:
import os, json, hashlib, importlib.metadata, platform
from pathlib import Path
from getpass import getpass
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi, get_token
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-3-270m-it"
DATASET_ID = "AmareshHebbar/clinical-summarizer-sft"
N_TRAIN_ROWS = 1500  # Match main.ipynb; lower for a quick first pass.
CONTEXT_LIMITS = [512, 1024, 2048, 4096, 8192]
BASELINE_OUTPUT_LIMITS = [512, 2048]
OUTPUT_DIR = Path("training_token_audit")
OUTPUT_DIR.mkdir(exist_ok=True)
assert N_TRAIN_ROWS > 0 and all(n > 0 for n in CONTEXT_LIMITS)

token = os.environ.get("HF_TOKEN") or get_token()
if not token:
    token = getpass("Hugging Face read token: ").strip()
if not token:
    raise RuntimeError("A token with Gemma access is required.")
os.environ["HF_TOKEN"] = token
api = HfApi(token=token)

# Prefer revisions from the experiment being investigated.
config_paths = [Path("baseline_evaluation_diagnostic_10/generation_config.json"),
                Path("baseline_evaluation/generation_config.json")]
saved = next((json.loads(p.read_text(encoding="utf-8")) for p in config_paths if p.exists()), {})
assert saved.get("model", MODEL_ID) == MODEL_ID
model_revision = saved.get("model_revision") or api.model_info(MODEL_ID).sha
dataset_revision = saved.get("dataset_revision") or api.dataset_info(DATASET_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=token, use_fast=True)
assert tokenizer.is_fast, "Offset mappings require the fast tokenizer."
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
train = load_dataset(DATASET_ID, revision=dataset_revision,
                     split=f"train[:{N_TRAIN_ROWS}]", token=token)
config = {"model": MODEL_ID, "model_revision": model_revision,
          "dataset": DATASET_ID, "dataset_revision": dataset_revision,
          "split": f"train[:{N_TRAIN_ROWS}]", "rows_loaded": len(train),
          "context_limits": CONTEXT_LIMITS, "truncation_side": tokenizer.truncation_side,
          "tokenization": "formatted text, tokenizer defaults; no packing",
          "transformers": importlib.metadata.version("transformers"),
          "datasets": importlib.metadata.version("datasets"), "python": platform.python_version()}
(OUTPUT_DIR / "audit_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))


In [5]:
def format_training_row(example):
    # Exactly the role folding used by main.ipynb; require its single-turn task.
    messages = example["messages"]
    assert [m["role"] for m in messages] in (
        ["system", "user", "assistant"], ["system", "user", "model"]
    ), "Unexpected conversation structure; audit this row separately."
    system, note, answer = [m["content"] for m in messages]
    prompt = f"{system}\n\n{note}".strip() if system else note
    user = [{"role": "user", "content": prompt}]
    prefix = tokenizer.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
    text = tokenizer.apply_chat_template(user + [{"role": "model", "content": answer}], tokenize=False)
    assert text.startswith(prefix) and text[len(prefix):].startswith(answer), "Template boundary changed."
    return text, prefix, answer


def retention(offsets, answer_start, answer_end, limit, side="right"):
    """Count tokens overlapping assistant content; exclude token 0 (causal shift)."""
    assert side in ("right", "left")
    positions = [i for i, (a, b) in enumerate(offsets)
                 if b > answer_start and a < answer_end and b > a]
    lo, hi = (0, min(limit, len(offsets))) if side == "right" else (max(0, len(offsets)-limit), len(offsets))
    # First retained token has no preceding logit in a causal-LM loss.
    retained = sum(lo < i < hi for i in positions)
    total = len(positions)
    return total, retained

# Boundary and causal-shift checks independent of downloaded data.
mock_offsets = [(0, 0), (0, 3), (3, 6), (6, 9), (9, 12)]
assert retention(mock_offsets, 6, 12, 3) == (2, 0)
assert retention(mock_offsets, 6, 12, 4) == (2, 1)
assert retention(mock_offsets, 6, 12, 5) == (2, 2)
assert retention(mock_offsets, 6, 12, 2, "left") == (2, 1)
print("Retention self-checks passed.")


In [6]:
rows, cutoff_rows, previews = [], [], []
for index, example in enumerate(train):
    text, prefix, answer = format_training_row(example)
    enc = tokenizer(text, truncation=False, return_offsets_mapping=True)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]
    start, end = len(prefix), len(prefix) + len(answer)
    answer_positions = [i for i, (a,b) in enumerate(offsets) if b > start and a < end and b > a]
    assert answer_positions, f"Row {index}: no answer tokens found."
    try:
        parsed = json.loads(answer)
        reference_error = None
        reference_object = isinstance(parsed, dict)
    except ValueError as exc:
        reference_error, reference_object = str(exc), False
    row = {"index": index, "messages_sha256": hashlib.sha256(
               json.dumps(example["messages"], sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
           "full_tokens": len(ids), "prompt_tokens_in_full_text": answer_positions[0],
           "answer_tokens": len(answer_positions),
           "answer_plus_end_tokens": len(ids) - answer_positions[0],
           "default_extra_tokens_vs_chat": len(ids) - len(tokenizer(text, add_special_tokens=False)["input_ids"]),
           "reference_valid_json": reference_error is None,
           "reference_is_object": reference_object, "reference_json_error": reference_error}
    rows.append(row)
    for limit in CONTEXT_LIMITS:
        total, kept = retention(offsets, start, end, limit, tokenizer.truncation_side)
        truncated_ids = tokenizer(text, truncation=True, max_length=limit)["input_ids"]
        expected = ids[:limit] if tokenizer.truncation_side == "right" else ids[-limit:]
        assert truncated_ids == expected, f"Row {index}: tokenizer truncation differs from simulation."
        cutoff_rows.append({"index": index, "context_limit": limit,
            "full_sequence_fits": len(ids) <= limit,
            "answer_tokens": total, "retained_answer_tokens": kept,
            "retained_answer_fraction": kept / total,
            "no_answer_supervision": kept == 0,
            "partial_answer": 0 < kept < total,
            "complete_answer_content": kept == total})
        if index < 3 and limit == 512:
            previews.append({"index": index, "context_limit": limit,
                "last_200_retained_tokens": tokenizer.decode(truncated_ids[-200:], skip_special_tokens=False),
                "first_120_reference_tokens": tokenizer.decode(ids[answer_positions[0]:answer_positions[0]+120], skip_special_tokens=False),
                "retained_answer_tokens": kept, "answer_tokens": total})
    if (index + 1) % 100 == 0:
        print(f"Audited {index + 1}/{len(train)}")
lengths, cutoffs = pd.DataFrame(rows), pd.DataFrame(cutoff_rows)
lengths.to_csv(OUTPUT_DIR / "per_example.csv", index=False)
cutoffs.to_csv(OUTPUT_DIR / "truncation_by_example.csv", index=False)
(OUTPUT_DIR / "truncation_previews.json").write_text(json.dumps(previews, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved per-example lengths, truncation counts, and three 512-token previews.")


In [7]:
summary = {"rows": len(lengths), "invalid_reference_json": int((~lengths.reference_valid_json).sum()),
           "non_object_references": int((~lengths.reference_is_object).sum()),
           "length_percentiles": {}, "context_limits": [], "reference_output_budgets": []}
for name in ("prompt_tokens_in_full_text", "answer_tokens", "full_tokens"):
    summary["length_percentiles"][name] = {str(q): float(lengths[name].quantile(q))
                                           for q in (0, .5, .9, .95, .99, 1)}
for limit, group in cutoffs.groupby("context_limit"):
    summary["context_limits"].append({"context_limit": int(limit), "examples": len(group),
        **{name: int(group[name].sum()) for name in (
            "full_sequence_fits", "no_answer_supervision", "partial_answer", "complete_answer_content")},
        "mean_retained_answer_fraction": float(group.retained_answer_fraction.mean()),
        "answer_token_retention_fraction": float(group.retained_answer_tokens.sum()/group.answer_tokens.sum())})
for budget in BASELINE_OUTPUT_LIMITS:
    summary["reference_output_budgets"].append({"max_new_tokens": budget,
        "references_fitting_including_end_marker": int((lengths.answer_plus_end_tokens <= budget).sum())})
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Token lengths (training prompt, without baseline schema):")
display(pd.DataFrame(summary["length_percentiles"]))
print("Truncation impact:")
display(pd.DataFrame(summary["context_limits"]))
print("Invalid reference JSON:", summary["invalid_reference_json"])
display(pd.DataFrame(summary["reference_output_budgets"]))
print("Rows whose 512-token training sequence contains no answer supervision:")
display(cutoffs.query("context_limit == 512 and no_answer_supervision").head(10))


## Interpret the results before changing the task

`no_answer_supervision` means no assistant content token can contribute to next-token loss in that truncated sequence. With full-text loss, the trainer may still learn the user note. `partial_answer` means the model can learn only a prefix of the target. `complete_answer_content` counts answer content; `full_sequence_fits` additionally requires the whole formatted conversation, including its closing tokens.

The reference-output budget counts whether the target could fit, not whether a model would generate it accurately. Baseline schema prompts add tokens and are deliberately excluded from this training audit. Invalid reference JSON is a separate data-quality problem; preserve these rows in the audit rather than silently repairing them.

Choose a feasible context based on complete input/answer lengths, then verify actual trainer labels and GPU memory before a pilot. A poor baseline does not establish that fine-tuning cannot work. If complete targets remain too expensive, select a smaller use case/dataset deliberately.

### Optional: verify the actual trainer's prepared examples

In the training notebook, split construction of `trainer` from `trainer.train()`. Run the following snippet **after constructing the trainer and before training**. It reports active labels and their text. Full-text labels include prompt tokens; inspect the end against the original reference. Some trainers build labels only in the collator, so the snippet uses that collator when necessary. Packing can change row correspondence and must be recorded separately.

```python
print(type(trainer.args).__name__, trainer.args)
prepared = [trainer.train_dataset[i] for i in range(min(3, len(trainer.train_dataset)))]
for i, example in enumerate(prepared):
    ids = example["input_ids"]
    labels = example.get("labels")
    if labels is None:
        batch = trainer.data_collator([example])
        ids = batch["input_ids"][0].tolist()
        labels = batch["labels"][0].tolist()
    active_ids = [token_id for position, (token_id, label) in enumerate(zip(ids, labels))
                  if position > 0 and label != -100]
    print("Example", i, "input tokens", len(ids), "active next-token labels", len(active_ids))
    print(tokenizer.decode(active_ids[-160:], skip_special_tokens=False))
```


In [8]:
# Package the audit artifacts for downloading from the remote runtime.
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR.parent,
                             base_dir=OUTPUT_DIR.name)
print("Download:", archive)
# In browser Colab only:
# from google.colab import files
# files.download(archive)
